# LRao full ablation (IID multi)

Three axes, one training per configuration:

1. **Early stopping vs full training** — each run tracks the published IID
   early-stop rule (10% val split, check every epoch, patience 3, min-delta
   0.5%); when ES *would* fire we record the epoch and save that (best-val)
   checkpoint, then **keep training** to the full budget → an ES model and a
   fully-trained model from the same run. Both are evaluated.
2. **Regularization sweep** — the Σ-cutoff, applied consistently in the LFI
   loss AND the detection statistic: `[1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-9, 0]`
   (0 = no regularization at all). Other regularizers fixed OFF (wd 0, no
   grad clip) so the cutoff is isolated.
3. **δ sweep** (numeric-derivative step of G in the statistic) — detection-only,
   re-scores the saved checkpoints at one reference cutoff: 5 values
   `[0.001, 0.005, 0.01, 0.05, 0.1]`, both ES and final models.

**Self-contained math:** the LFI loss and scorer are defined IN THIS NOTEBOOK
with explicit `cutoff`/`delta` arguments — no dependence on the repo's current
LRao signatures (immune to clone-state defaults).

Recipe otherwise: robust median/IQR front-end, [128] ReLU, batch 512,
Adam 5e-4, 1000 epochs, seeds 42–46, published pools/planting, Pd@Pfa=0.1.
Test metrics snapshotted every 50 epochs (dynamics; no selection).

**Budget:** 7 cutoffs × 8 n × 5 seeds = 280 trainings ≈ **9–11 h on a T4**
(≈3 h on an A100). Fully resume-safe — run over several sessions; the
analysis cells work on partial results. To split the work, set
`SEEDS = [42, 43, 44]` first and extend later. The δ sweep is ~15 min of
pure re-scoring. Checkpoints (ES + final per run) ≈ 60 MB total.

In [ ]:
!git clone -b rebuttal --depth 1 https://github.com/michaelpiro/final-paper-experiment.git repo
%cd repo
import os, sys, json, time, copy, contextlib
import numpy as np
import torch
sys.path.insert(0, '.')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')
assert os.path.exists('repro/data/pavia-u.mat')

In [ ]:
# ----------------- knobs -----------------
N_LIST   = [20, 40, 60, 100, 200, 500, 1000, 2000]
SEEDS    = [42, 43, 44, 45, 46]
CUTOFFS  = [1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-9, 0.0]   # 0.0 = no reg
DELTAS   = [0.001, 0.005, 0.01, 0.05, 0.1]             # detection-only sweep
DELTA_REF_CUTOFF = 1e-5        # checkpoints re-scored in the delta sweep
MAX_EPOCHS = 1000
SNAP = 50                      # test-metric snapshots (dynamics only)
# published IID early-stop rule (tracked, not enforced):
VAL_FRACTION, PATIENCE, MIN_DELTA = 0.1, 3, 0.005
OUT = 'results/lrao_ablation'

def clab(c):
    return 'none' if c == 0.0 else f'{c:.0e}'

# published multi reference lines (verified camera-ready, Pd@0.1)
REF_N    = [20, 40, 60, 100, 200, 500, 1000, 2000]
REF_DART = [0.287, 0.300, 0.327, 0.390, 0.402, 0.447, 0.537, 0.595]
REF_LRAO = [0.329, 0.367, 0.465, 0.501, 0.509, 0.565, 0.561, 0.553]

In [ ]:
# ----------------- protocol + self-contained LRao math -----------------
import yaml
from tqdm.auto import tqdm
from repro.protocols.iid import load_hsi, build_pools, _pd_at_fa, _auc
from repro.core.data import plant_targets
from repro.core.models import ScoreNet
from repro.core.normalization import robust_whitening_iqr

cfg = yaml.safe_load(open('repro/configs/iid_multi.yaml'))
cfg.update(dataset='repro/data/pavia-u.mat')


def build_data(seed):
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    data, gt = load_hsi(cfg['dataset'])
    bkg, tgt = build_pools(data, gt.flatten(), cfg, 'multi')
    s = tgt.mean(axis=0).astype(np.float32)
    idx = np.arange(len(bkg)); rng.shuffle(idx)
    shuf = bkg[idx]
    pool = shuf[:max(N_LIST)].astype(np.float32)
    te = shuf[-int(cfg['test_size']):].astype(np.float32)
    planted, labels, _ = plant_targets(te, s, cfg['amplitude'],
                                       cfg['target_fraction'],
                                       model='additive', seed=seed)
    return pool, planted.astype(np.float32), labels, s


def lfi_loss(model, batch, cutoff, detach_sigma=True):
    """LFI cost -tr(G^T Sigma^-1 G) with an EXPLICIT relative SVD cutoff
    (0.0 = full pseudo-inverse). Defined locally on purpose."""
    n = len(batch)
    ctx = torch.no_grad() if detach_sigma else contextlib.nullcontext()
    with ctx:
        psi0 = model(batch)
        mu = psi0.mean(0)
        c = psi0 - mu
        Sigma = (c.T @ c) / max(n - 1, 1)
        U, S, Vh = torch.linalg.svd(Sigma)
        thr = float(cutoff) * S[0]
        S_inv = torch.where(S > thr, 1.0 / S, torch.zeros_like(S))
        Sigma_inv = Vh.T @ torch.diag(S_inv) @ U.T
    from torch.func import jacrev, vmap
    J = vmap(jacrev(lambda x: model(x.unsqueeze(0)).squeeze(0)))(batch)
    G = J.mean(0)
    return -(G.T @ Sigma_inv @ G).trace()


@torch.no_grad()
def lrao_score(model, train_data, test_data, s, cutoff, delta):
    """Mode-2 statistic with explicit cutoff and finite-difference delta."""
    model.eval()
    d = train_data.shape[1]
    X_tr = torch.tensor(train_data, dtype=torch.float32, device=DEVICE)
    X_te = torch.tensor(test_data, dtype=torch.float32, device=DEVICE)
    I_d = torch.eye(d, device=DEVICE)
    psi_tr = model(X_tr).cpu().numpy().astype(np.float64)
    if not np.all(np.isfinite(psi_tr)):
        return np.zeros(len(test_data))
    mu = psi_tr.mean(0)
    Sigma = (psi_tr - mu).T @ (psi_tr - mu) / max(len(train_data) - 1, 1)
    U, S, Vh = np.linalg.svd((Sigma + Sigma.T) / 2)
    thr = float(cutoff) * S[0]
    S_inv = np.where(S > thr, 1.0 / S, 0.0)
    Sigma_inv = Vh.T @ np.diag(S_inv) @ U.T
    G = np.zeros((psi_tr.shape[1], d))
    for j in range(d):
        plus = model(X_tr + delta * I_d[j]).cpu().numpy()
        minus = model(X_tr - delta * I_d[j]).cpu().numpy()
        G[:, j] = ((plus - minus) / (2.0 * delta)).mean(0)
    g_s = G @ np.asarray(s, np.float64)
    J_s = float(g_s @ Sigma_inv @ g_s)
    psi_te = model(X_te).cpu().numpy()
    return (psi_te - mu) @ (Sigma_inv @ g_s) / np.sqrt(max(J_s, 1e-12))


def metrics(labels, sc):
    return dict(pd=_pd_at_fa(labels, sc, cfg['pfa']), auc=_auc(labels, sc))

In [ ]:
# ----------------- trainer: one run -> ES model + fully trained model -----------------
def train_one(tr, cutoff, seed, label, planted, labels, s, ckpt_dir):
    torch.manual_seed(seed)                              # published order:
    W = robust_whitening_iqr(tr)                         # W from FULL subset,
    net = ScoreNet(tr.shape[1], [128], 'relu',           # net built,
                   whitening=W).to(DEVICE)
    opt = torch.optim.Adam(net.parameters(), lr=cfg['lr'], weight_decay=0.0)
    X = torch.tensor(tr).to(DEVICE)
    n_val = max(1, int(len(X) * VAL_FRACTION))           # then val split
    perm0 = torch.randperm(len(X))
    Xf, Xv = X[perm0[:-n_val]], X[perm0[-n_val:]]
    N = len(Xf); bs = min(int(cfg['batch_size']), N)
    best_val, best_state, best_ep, bad = float('inf'), None, 0, 0
    es_epoch, es_state = None, None                      # where ES would fire
    val_hist, snaps = [], []
    pbar = tqdm(range(1, MAX_EPOCHS + 1), desc=label, leave=False)
    for ep in pbar:
        net.train()
        perm = torch.randperm(N)
        for i in range(0, N, bs):
            try:
                loss = lfi_loss(net, Xf[perm[i:i + bs]], cutoff)
            except Exception:
                continue
            if not torch.isfinite(loss):
                continue
            opt.zero_grad(); loss.backward(); opt.step()
        net.eval()
        try:
            vl = float(lfi_loss(net, Xv, cutoff).detach())
        except Exception:
            vl = float('nan')
        val_hist.append(vl)
        if np.isfinite(vl) and vl < best_val - MIN_DELTA * abs(best_val):
            best_val, best_ep, bad = vl, ep, 0
            best_state = copy.deepcopy(net.state_dict())
        else:
            bad += 1
        if es_epoch is None and bad >= PATIENCE:
            es_epoch = best_ep                           # ES fires HERE ...
            es_state = copy.deepcopy(best_state)         # ... but we go on
        if ep % SNAP == 0:
            sc = lrao_score(net, tr, planted, s, cutoff, 0.01)
            snaps.append({'epoch': ep, **metrics(labels, sc)})
            pbar.set_postfix(pd=f"{snaps[-1]['pd']:.3f}",
                             es=es_epoch if es_epoch else '-')
    if es_epoch is None:                                 # never fired
        es_epoch, es_state = best_ep, copy.deepcopy(best_state)
        never_stopped = True
    else:
        never_stopped = False
    final_state = copy.deepcopy(net.state_dict())
    os.makedirs(ckpt_dir, exist_ok=True)
    torch.save({'state_dict': {k: v.cpu() for k, v in es_state.items()},
                'epoch': es_epoch},
               os.path.join(ckpt_dir, label + '_es.pt'))
    torch.save({'state_dict': {k: v.cpu() for k, v in final_state.items()},
                'epoch': MAX_EPOCHS},
               os.path.join(ckpt_dir, label + '_final.pt'))
    out = {'es_epoch': es_epoch, 'never_stopped': never_stopped,
           'val_hist': [round(v, 6) for v in val_hist], 'snaps': snaps}
    for kind, state in (('es', es_state), ('final', final_state)):
        net.load_state_dict(state); net.eval()
        sc = lrao_score(net, tr, planted, s, cutoff, 0.01)
        out[kind] = metrics(labels, sc)
    return out

In [ ]:
# ----------------- experiment 1+2: cutoff x n x seed (resume-safe) -----------------
os.makedirs(OUT, exist_ok=True)
met_path = os.path.join(OUT, 'metrics.json')
rec = json.load(open(met_path)) if os.path.exists(met_path) else {}
rec['_meta'] = dict(n_list=N_LIST, seeds=SEEDS,
                    cutoffs=[clab(c) for c in CUTOFFS],
                    max_epochs=MAX_EPOCHS, val_fraction=VAL_FRACTION,
                    patience=PATIENCE, min_delta=MIN_DELTA)
t0 = time.time()
for seed in SEEDS:
    pool, planted, labels, s = build_data(seed)
    for n in N_LIST:
        tr = pool[:n]
        for co in CUTOFFS:
            key = f'c{clab(co)}_n{n}_s{seed}'
            if rec.get(key, {}).get('final'):
                continue
            t1 = time.time()
            r = train_one(tr, co, seed, key, planted, labels, s,
                          os.path.join(OUT, 'ckpt'))
            r['sec'] = round(time.time() - t1)
            rec[key] = r
            json.dump(rec, open(met_path, 'w'))
            print(f"{key}: ES ep{r['es_epoch']}"
                  f"{' (never fired)' if r['never_stopped'] else ''} "
                  f"Pd_es={r['es']['pd']:.3f} Pd_final={r['final']['pd']:.3f} "
                  f"({r['sec']}s)", flush=True)
print(f'TOTAL {(time.time() - t0) / 3600:.2f} h')

In [ ]:
# ----------------- experiment 3: delta sweep (re-scores saved ckpts) -----------------
dl_path = os.path.join(OUT, 'delta_metrics.json')
drec = json.load(open(dl_path)) if os.path.exists(dl_path) else {}
for seed in SEEDS:
    pool, planted, labels, s = build_data(seed)
    for n in N_LIST:
        tr = pool[:n]
        for kind in ('es', 'final'):
            ck = os.path.join(OUT, 'ckpt',
                              f'c{clab(DELTA_REF_CUTOFF)}_n{n}_s{seed}_{kind}.pt')
            if not os.path.exists(ck):
                continue
            blob = torch.load(ck, map_location='cpu', weights_only=False)
            net = ScoreNet(tr.shape[1], [128], 'relu',
                           whitening=robust_whitening_iqr(tr)).to(DEVICE)
            net.load_state_dict(blob['state_dict']); net.eval()
            for delta in DELTAS:
                key = f'd{delta}_n{n}_s{seed}_{kind}'
                if key in drec:
                    continue
                sc = lrao_score(net, tr, planted, s, DELTA_REF_CUTOFF, delta)
                drec[key] = metrics(labels, sc)
                json.dump(drec, open(dl_path, 'w'))
        print(f'delta sweep: n={n} seed={seed} done', flush=True)
print('delta sweep complete')

In [ ]:
# ----------------- analysis + figures (works on partial results) -----------------
import matplotlib.pyplot as plt
from IPython.display import Image, display

FIG = os.path.join(OUT, 'figures'); os.makedirs(FIG, exist_ok=True)
rec = json.load(open(met_path))
CL = [clab(c) for c in CUTOFFS]

P = {k: np.full((len(CL), len(N_LIST), len(SEEDS)), np.nan)
     for k in ('es', 'final')}
E = np.full((len(CL), len(N_LIST), len(SEEDS)), np.nan)   # es epoch
for i, cl in enumerate(CL):
    for j, n in enumerate(N_LIST):
        for k, sd in enumerate(SEEDS):
            r = rec.get(f'c{cl}_n{n}_s{sd}')
            if not r or 'final' not in r:
                continue
            P['es'][i, j, k] = r['es']['pd']
            P['final'][i, j, k] = r['final']['pd']
            E[i, j, k] = r['es_epoch']

def show(fig, name):
    fig.tight_layout()
    p = os.path.join(FIG, name + '.png')
    fig.savefig(p, dpi=200); fig.savefig(p.replace('.png', '.pdf'))
    plt.close(fig); display(Image(p, width=980))

# --- fig 1: Pd vs n per cutoff, ES vs final panels ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
cols = plt.cm.viridis(np.linspace(0, 0.9, len(CL)))
for ax, kind in zip(axes, ('es', 'final')):
    for i, cl in enumerate(CL):
        ax.plot(N_LIST, np.nanmean(P[kind][i], axis=1), 'o-', color=cols[i],
                lw=1.5, label=f'cutoff {cl}')
    ax.plot(REF_N, REF_DART, 's--', color='tab:red', lw=1.4, label='DART (pub)')
    ax.plot(REF_N, REF_LRAO, 'x-', color='k', lw=1, label='LRao (pub)')
    ax.set_xscale('log'); ax.set_xticks(N_LIST); ax.set_xticklabels(N_LIST)
    ax.minorticks_off(); ax.grid(alpha=0.3)
    ax.set_xlabel('n'); ax.set_title(f'{kind.upper()} model')
axes[0].set_ylabel('Pd@0.1 (mean over seeds)')
axes[1].legend(fontsize=7, ncol=2)
fig.suptitle('LRao ablation: cutoff x n — early-stopped vs fully trained')
show(fig, 'cutoff_es_vs_final')

# --- fig 2: ES benefit (es - final) + es epoch ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for i, cl in enumerate(CL):
    axes[0].plot(N_LIST, np.nanmean(P['es'][i] - P['final'][i], axis=1),
                 'o-', color=cols[i], lw=1.4, label=f'cutoff {cl}')
axes[0].axhline(0, color='gray', lw=1)
axes[0].set_xscale('log'); axes[0].set_xticks(N_LIST)
axes[0].set_xticklabels(N_LIST); axes[0].minorticks_off()
axes[0].set_xlabel('n'); axes[0].set_ylabel('Pd(ES) - Pd(final)')
axes[0].set_title('what early stopping buys'); axes[0].grid(alpha=0.3)
axes[0].legend(fontsize=7)
im = axes[1].imshow(np.nanmean(E, axis=2), aspect='auto', cmap='magma')
axes[1].set_xticks(range(len(N_LIST))); axes[1].set_xticklabels(N_LIST)
axes[1].set_yticks(range(len(CL))); axes[1].set_yticklabels(CL, fontsize=8)
axes[1].set_xlabel('n'); axes[1].set_ylabel('cutoff')
axes[1].set_title('mean ES epoch')
plt.colorbar(im, ax=axes[1], fraction=0.046)
show(fig, 'es_benefit_and_epoch')

# --- fig 3: delta sweep ---
if os.path.exists(dl_path):
    drec = json.load(open(dl_path))
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
    ncols = plt.cm.plasma(np.linspace(0, 0.8, len(N_LIST)))
    for ax, kind in zip(axes, ('es', 'final')):
        for j, n in enumerate(N_LIST):
            vals = [np.nanmean([drec[f'd{d}_n{n}_s{sd}_{kind}']['pd']
                                for sd in SEEDS
                                if f'd{d}_n{n}_s{sd}_{kind}' in drec] or [np.nan])
                    for d in DELTAS]
            ax.plot(DELTAS, vals, 'o-', color=ncols[j], lw=1.4, label=f'n={n}')
        ax.set_xscale('log'); ax.grid(alpha=0.3)
        ax.set_xlabel('delta (finite-difference step)')
        ax.set_title(f'{kind.upper()} model (cutoff {clab(DELTA_REF_CUTOFF)})')
    axes[0].set_ylabel('Pd@0.1'); axes[1].legend(fontsize=7, ncol=2)
    fig.suptitle('delta sensitivity of the detection statistic')
    show(fig, 'delta_sweep')

# --- summary table ---
lines = ['# LRao ablation — Pd@0.1 mean over seeds', '']
for kind in ('es', 'final'):
    lines += [f'## {kind}', '| cutoff | ' + ' | '.join(f'n={n}' for n in N_LIST) + ' |',
              '|' + '---|' * (len(N_LIST) + 1)]
    for i, cl in enumerate(CL):
        lines.append(f'| {cl} | ' + ' | '.join(
            f'{v:.3f}' if np.isfinite(v) else '—'
            for v in np.nanmean(P[kind][i], axis=1)) + ' |')
    lines.append('')
open(os.path.join(OUT, 'summary.md'), 'w').write('\n'.join(lines))
print('\n'.join(lines))

In [ ]:
# ----------------- zip -----------------
!zip -qr lrao_ablation_light.zip results/lrao_ablation -x "*/ckpt/*"
!zip -qr lrao_ablation_full.zip results/lrao_ablation
!ls -lh lrao_ablation_*.zip
try:
    from google.colab import files
    files.download('lrao_ablation_light.zip')
except Exception as e:
    print('manual download:', e)